<img src="https://raw.githubusercontent.com/ComplianceAnalytics/aml-book1/main/assets/cal_logo_banner.png" alt="Compliance Analytics Ltd" width="300" onerror="this.style.display='none'">

# Applied AML Analytics: Turning Data Science Skills into Compliance Decisions
## Chapter 3 — The Transaction Monitoring Lifecycle
### Companion Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ComplianceAnalytics/aml-book1/blob/main/notebooks/chapter_03.ipynb)

---

**Book:** *Applied AML Analytics: Turning Data Science Skills into Compliance Decisions* — Book 1  
**Publisher companion repository:** [github.com/ComplianceAnalytics/aml-book1](https://github.com/ComplianceAnalytics/aml-book1)  
**Dataset:** Northgate Retail Bank (synthetic — all data is fictional)  
**Chapters covered:** 3 (this notebook) · 4 · 5 · 6 · 7 · 8  

> **How to use this notebook**  
> Run cells top-to-bottom using **Shift+Enter** or the ▶ button. The setup cell (Section 0) must run first — it generates the Northgate dataset that all later cells depend on. You do not need to install anything; all required libraries are pre-installed in Google Colab.

---

## Contents

| Section | Description | Exercise link |
|---------|-------------|---------------|
| **0. Setup** | Generate the Northgate dataset | — |
| **1. Colab Preview** | First look at the data (mirrors Section 3.10 of the text) | — |
| **2. Exercise 3.1 — Part A Extension** | Mule account profile · data quality gaps · lifecycle risk | Exercise 3.1 Part A |
| **3. Reflection cells** | Structured answer prompts | Exercise 3.1 Parts A–C |

---
## Section 0 — Setup: Generate the Northgate Dataset

**Run this cell first.** It generates four CSV files in the Colab session's working directory:

| File | Rows | Description |
|------|------|-------------|
| `nb_transactions.csv` | ~23,000 | All account transactions, Jan–Dec 2023 |
| `nb_customers.csv` | 500 | Customer and account records |
| `nb_counterparties.csv` | 300 | Counterparty firms and their country codes |
| `nb_accounts.csv` | 500 | Account metadata (same as customers for this dataset) |

The dataset is **fully synthetic**. Northgate Retail Bank does not exist. All account IDs, names, and transactions are generated from a fixed random seed and are used solely for educational purposes.

In [ ]:
import numpy as np
import pandas as pd
from datetime import date, timedelta

rng = np.random.default_rng(42)

# ── Counterparties ────────────────────────────────────────────────────────────
HIGH_RISK  = ['KP', 'IR', 'MM', 'SY', 'YE', 'AF', 'LY']
LOW_RISK   = ['US', 'GB', 'DE', 'FR', 'CA', 'AU', 'SG', 'JP', 'NL', 'CH']

n_cpty      = 300
cpty_ids    = [f'CPT{i:04d}' for i in range(1, n_cpty + 1)]
cpty_cc     = (rng.choice(HIGH_RISK, size=30).tolist() +
               rng.choice(LOW_RISK,  size=270, replace=True).tolist())
rng.shuffle(cpty_cc)
df_cpty = pd.DataFrame({'counterparty_id': cpty_ids, 'country_code': cpty_cc})
df_cpty.to_csv('nb_counterparties.csv', index=False)

# ── Customers & Accounts ──────────────────────────────────────────────────────
n_cust   = 500
cust_ids = [f'NRB_{i:03d}' for i in range(1, n_cust + 1)]
acct_ids = [f'ACC{i:04d}' for i in range(1, n_cust + 1)]
mule_idx = list(range(6))   # ACC0001–ACC0006 are mule accounts

occupations = ['Employed', 'Self-Employed', 'Retired', 'Student', None]
occ_probs   = [0.55, 0.20, 0.12, 0.08, 0.05]
crr_scores  = rng.choice([1,2,3,4,5], p=[0.35,0.30,0.20,0.10,0.05], size=n_cust)
for i in mule_idx:
    crr_scores[i] = rng.choice([3,4])
incomes_k = rng.lognormal(mean=3.1, sigma=0.5, size=n_cust) * 1000
for i in mule_idx:
    incomes_k[i] = rng.uniform(18, 24) * 1000

df_cust = pd.DataFrame({
    'customer_id':       cust_ids,
    'account_id':        acct_ids,
    'occupation':        rng.choice(occupations, p=occ_probs, size=n_cust),
    'crr_score':         crr_scores,
    'stated_income_usd': np.round(incomes_k, -2),
    'account_open_date': [
        (date(2020,1,1) + timedelta(days=int(d))).isoformat()
        for d in rng.integers(0, 1460, size=n_cust)
    ],
})
df_cust.to_csv('nb_customers.csv', index=False)
df_cust.to_csv('nb_accounts.csv',  index=False)

# ── Transactions ──────────────────────────────────────────────────────────────
txn_rows = []
start    = date(2023, 1, 1)
txn_id   = 1

for i, (cid, aid) in enumerate(zip(cust_ids, acct_ids)):
    is_mule = i in mule_idx
    if is_mule:
        for m in range(12):
            for _ in range(rng.integers(3, 9)):
                day      = rng.integers(1, 28)
                txn_date = date(2023, m + 1, day)
                amount   = round(rng.uniform(7800, 9800), 2)
                cpty     = rng.choice(cpty_ids[:30])   # high-risk counterparties
                txn_rows.append({'txn_id': f'TXN{txn_id:06d}', 'account_id': aid,
                                 'txn_date': txn_date.isoformat(), 'txn_type': 'CASH_IN',
                                 'amount': amount, 'counterparty_id': cpty})
                txn_id += 1
    else:
        for _ in range(rng.integers(12, 80)):
            txn_date = start + timedelta(days=int(rng.integers(0, 365)))
            txn_type = rng.choice(['CASH_IN','TRANSFER_OUT','TRANSFER_IN','CARD'],
                                   p=[0.15, 0.35, 0.35, 0.15])
            amount   = round(min(rng.lognormal(6.5, 1.2), 50000), 2)
            cpty_pool = cpty_ids[30:] if rng.random() > 0.03 else cpty_ids[:30]
            txn_rows.append({'txn_id': f'TXN{txn_id:06d}', 'account_id': aid,
                             'txn_date': txn_date.isoformat(), 'txn_type': txn_type,
                             'amount': amount, 'counterparty_id': rng.choice(cpty_pool)})
            txn_id += 1

df_txn = (pd.DataFrame(txn_rows)
            .assign(txn_date=lambda d: pd.to_datetime(d['txn_date']))
            .sort_values('txn_date')
            .reset_index(drop=True))
df_txn.to_csv('nb_transactions.csv', index=False)

print(f"✅ Dataset generated")
print(f"   nb_counterparties : {len(df_cpty):>6,} rows")
print(f"   nb_customers      : {len(df_cust):>6,} rows")
print(f"   nb_transactions   : {len(df_txn):>6,} rows")
print(f"   Date range        : {df_txn['txn_date'].min().date()} → {df_txn['txn_date'].max().date()}")

---
## Section 1 — Colab Preview: A First Look at the Data

> *This section mirrors Section 3.10 of the textbook exactly. The code here is the same code printed in the blue "Colab Preview" box. Run it to see the real output.*

Before building rules, you need to know your data. The two cells below load the Northgate dataset and give you a first hands-on look at its structure. In Chapter 4, you will write the first formal rule on top of this foundation.

In [ ]:
import pandas as pd

# Load the Northgate synthetic dataset
df_txn  = pd.read_csv('nb_transactions.csv', parse_dates=['txn_date'])
df_cust = pd.read_csv('nb_customers.csv')

# Dataset summary — how much data are we working with?
print(f"Transactions : {len(df_txn):>8,}")
print(f"Accounts     : {df_txn['account_id'].nunique():>8,}")
print(f"Date range   : {df_txn['txn_date'].min().date()} to {df_txn['txn_date'].max().date()}")

In [ ]:
# Preview the first few rows (key columns)
df_txn[['account_id', 'txn_date', 'txn_type', 'amount', 'counterparty_id']].head(5)

**What you're seeing:** 23,188 transactions across 500 accounts for calendar year 2023. Most are ordinary retail activity — transfers, card payments, small cash deposits. The date column is a proper datetime object, the amount is a float, and each transaction links to a counterparty ID that we can join to `nb_counterparties.csv` to get the counterparty's country.

The transaction type breakdown tells you something important about the population:

In [ ]:
# Transaction type breakdown
type_summary = (
    df_txn.groupby('txn_type')
          .agg(count=('txn_id','count'), total_amount=('amount','sum'), avg_amount=('amount','mean'))
          .assign(pct_count=lambda d: (d['count'] / d['count'].sum() * 100).round(1),
                  total_amount=lambda d: d['total_amount'].round(0),
                  avg_amount=lambda d: d['avg_amount'].round(0))
)
print(type_summary.to_string())

---
## Section 2 — Exercise 3.1 Part A Extension: The Mule Account Profile

> *This section is the "Colab Extension" described in the Exercise 3.1 Part A box in the textbook. Complete the cells below, then use your findings to answer the reflection questions in Section 3.*

Six accounts in the Northgate dataset are **mule accounts** — part of the fictional Northgate structuring network. Their account IDs are `ACC0001` through `ACC0006`. The cells below give you a pre-filtered view of these accounts so you can examine their behaviour and identify which stage of the TM lifecycle is most at risk given the data quality gaps in the dataset.

Work through each cell. At the end of each block there is a `# ✏️ YOUR OBSERVATION:` comment — use it to write your own notes before moving to the reflection cells in Section 3.

In [ ]:
# The six mule accounts
MULE_IDS = [f'ACC{i:04d}' for i in range(1, 7)]

# Filter transactions to mule accounts only
df_mule_txn  = df_txn[df_txn['account_id'].isin(MULE_IDS)].copy()
df_mule_cust = df_cust[df_cust['account_id'].isin(MULE_IDS)].copy()

print(f"Mule account transactions : {len(df_mule_txn):,}")
print(f"Mule accounts             : {df_mule_txn['account_id'].nunique()}")
print()
print("Customer profile:")
print(df_mule_cust[['account_id','occupation','crr_score','stated_income_usd','account_open_date']].to_string(index=False))

**Look at the customer profile above.** Notice the stated income and CRR scores. Before you run the next cell, form a hypothesis: does the customer profile alone suggest anything suspicious?

```
# ✏️ YOUR OBSERVATION — customer profile:
#
#
```

In [ ]:
# Monthly cash-in totals per mule account
monthly = (
    df_mule_txn[df_mule_txn['txn_type'] == 'CASH_IN']
    .assign(month=lambda d: d['txn_date'].dt.to_period('M'))
    .groupby(['account_id','month'])
    .agg(n_deposits=('amount','count'), total_cash_in=('amount','sum'), avg_deposit=('amount','mean'))
    .round(2)
)
print("Monthly cash-in summary per mule account:")
print(monthly.to_string())

**What you're seeing:** Each mule account makes 3–8 cash deposits per month, each just below USD 10,000. The total monthly cash-in for a single mule account — roughly USD 50,000–90,000 — far exceeds the stated income of USD 18,000–24,000 per year.

This is the structuring red flag: deliberately keeping individual deposits below the Currency Transaction Report threshold of USD 10,000 to avoid automatic reporting.

```
# ✏️ YOUR OBSERVATION — monthly pattern:
# Which account has the highest monthly cash-in? What is the ratio of monthly cash-in
# to stated annual income? What does this tell a Level 2 investigator?
#
#
```

In [ ]:
import matplotlib.pyplot as plt

# Distribution of individual deposit amounts — all mule accounts combined
cash_in = df_mule_txn[df_mule_txn['txn_type'] == 'CASH_IN']['amount']

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(cash_in, bins=20, color='#4472C4', edgecolor='white', alpha=0.85)
ax.axvline(10_000, color='#E74C3C', linewidth=2, linestyle='--', label='CTR threshold (USD 10,000)')
ax.set_xlabel('Individual deposit amount (USD)', fontsize=11)
ax.set_ylabel('Number of transactions', fontsize=11)
ax.set_title('Mule Account Cash Deposit Distribution\n(all 6 accounts, full year 2023)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print(f"\nDeposit range : USD {cash_in.min():,.2f} – USD {cash_in.max():,.2f}")
print(f"Mean deposit  : USD {cash_in.mean():,.2f}")
print(f"Max deposit   : USD {cash_in.max():,.2f}  (threshold: USD 10,000)")
print(f"All deposits below threshold? {(cash_in < 10_000).all()}")

**What you're seeing:** Every single deposit is below USD 10,000 — and they cluster between USD 7,800 and USD 9,800. This is the geometric signature of structuring. A random retail customer's deposits would be much more spread out across the amount range.

```
# ✏️ YOUR OBSERVATION — amount distribution:
# If you were the Level 1 analyst reviewing one of these alerts, what would the
# deposit histogram tell you that the raw transaction list might not make obvious?
#
#
```

In [ ]:
# Data quality check — what fields are missing for the mule accounts?
print("=== DATA QUALITY ASSESSMENT — Mule accounts ===")
print()

# Customer fields
print("Customer record completeness:")
for col in df_mule_cust.columns:
    n_missing = df_mule_cust[col].isna().sum()
    pct = n_missing / len(df_mule_cust) * 100
    status = "⚠️  MISSING" if n_missing > 0 else "✅ complete"
    print(f"  {col:<25} {status}  ({n_missing}/{len(df_mule_cust)} records)")

print()

# Transaction fields
print("Transaction record completeness (mule accounts only):")
for col in df_mule_txn.columns:
    n_missing = df_mule_txn[col].isna().sum()
    pct = n_missing / len(df_mule_txn) * 100
    status = "⚠️  MISSING" if n_missing > 0 else "✅ complete"
    print(f"  {col:<25} {status}  ({n_missing}/{len(df_mule_txn)} records)")

print()

# Check income vs actual cash-in
annual_cash = (
    df_mule_txn[df_mule_txn['txn_type'] == 'CASH_IN']
    .groupby('account_id')['amount'].sum()
    .rename('annual_cash_in')
)
income_check = df_mule_cust.set_index('account_id')[['stated_income_usd']].join(annual_cash)
income_check['ratio_cash_to_income'] = (income_check['annual_cash_in'] / income_check['stated_income_usd']).round(1)
print("Income vs actual cash-in (annual):")
print(income_check.to_string())

**Key finding:** The cash-in to income ratio for mule accounts ranges from roughly 25× to 45× stated annual income. This is a critical data-based red flag that would be visible to a well-designed Rule 1 alert — but only if the stated income field is populated and trusted.

Notice also whether any customer fields are missing. A missing occupation field is not just a gap — it tells you something about the onboarding process and whether the account's risk profile can be properly assessed at the Data Collection stage of the TM lifecycle.

```
# ✏️ YOUR OBSERVATION — data quality:
# Which stage of the TM lifecycle is most at risk given these data quality gaps?
# Support your answer with at least two specific observations from the cells above.
#
#
```

In [ ]:
# Counterparty country analysis for mule accounts
df_cpty = pd.read_csv('nb_counterparties.csv')
HIGH_RISK_COUNTRIES = ['KP', 'IR', 'MM', 'SY', 'YE', 'AF', 'LY']

mule_cpty = (
    df_mule_txn
    .merge(df_cpty, on='counterparty_id', how='left')
    .assign(is_high_risk=lambda d: d['country_code'].isin(HIGH_RISK_COUNTRIES))
)

print("Counterparty risk profile — mule accounts:")
risk_summary = mule_cpty.groupby(['account_id','is_high_risk']).size().unstack(fill_value=0)
risk_summary.columns = ['low_risk_cpty_txns', 'high_risk_cpty_txns']
risk_summary['pct_high_risk'] = (
    risk_summary['high_risk_cpty_txns'] /
    risk_summary.sum(axis=1) * 100
).round(1)
print(risk_summary.to_string())
print()
print("High-risk countries transacted with:")
print(mule_cpty[mule_cpty['is_high_risk']].groupby('account_id')['country_code']
      .apply(lambda x: ', '.join(sorted(set(x)))).to_string())

---
## Section 3 — Reflection: Exercise 3.1 Answer Cells

> *Use the cells below to record your answers to Exercise 3.1. There are no right or wrong wordings — the goal is to demonstrate that you can apply the TM lifecycle framework to a real data profile.*

### Part A — Tracing the Northgate Scenario Through the TM Lifecycle

For each of the six TM lifecycle stages, answer the three questions from the exercise:
1. What data is used, and what data quality risks exist?
2. What decision or output is produced, and what criteria govern it?
3. What could go wrong that would prevent correct identification or reporting?

#### Stage 1 — Data Collection

*(Edit this cell to write your answer)*

**Data used:**  

**Data quality risks (support with observations from Section 2):**  

**Decision/output produced:**  

**What could go wrong:**  

#### Stage 2 — Scenario Development

*(Edit this cell to write your answer)*

**Data used:**  

**Decisions/output produced (what threshold, what window?):**  

**What could go wrong:**  

#### Stage 3 — Alert Generation

*(Edit this cell to write your answer)*

**Data used:**  

**Output produced (what does the alert contain?):**  

**What could go wrong:**  

#### Stage 4 — Alert Review (L1)

*(Edit this cell to write your answer)*

**Data used:**  

**Decision/output produced:**  

**What could go wrong (consider alert fatigue):**  

#### Stage 5 — Case Investigation (L2/L3)

*(Edit this cell to write your answer)*

**Data used:**  

**Decision/output produced:**  

**What could go wrong:**  

#### Stage 6 — SAR Filing or Closure

*(Edit this cell to write your answer)*

**Data used:**  

**Decision/output produced:**  

**What could go wrong (consider tipping-off risk):**  

#### Synthesis — Which Stage Is Most at Risk?

*Based on your observations in Section 2, which stage of the TM lifecycle is most at risk given the data quality gaps visible in the Northgate dataset? Support your answer with at least two specific data points from the cells above.*

*(Edit this cell to write your answer)*

**Most at-risk stage:**  

**Evidence point 1:**  

**Evidence point 2:**  

**Why this matters operationally:**  

---
## What's Next

In Chapter 4, you will write `apply_rule_1()` — the formal Python implementation of the structuring rule — and run it against this dataset. You will see which accounts it catches, how the alert volume changes with different thresholds, and what a tuning decision looks like in practice.

| Chapter | Notebook | Topic |
|---------|----------|-------|
| 4 | [chapter_04.ipynb](https://colab.research.google.com/github/ComplianceAnalytics/aml-book1/blob/main/notebooks/chapter_04.ipynb) | Rule 1 — Cash Threshold Structuring |
| 5 | [chapter_05.ipynb](https://colab.research.google.com/github/ComplianceAnalytics/aml-book1/blob/main/notebooks/chapter_05.ipynb) | K-Means Customer Segmentation |
| 6 | [chapter_06.ipynb](https://colab.research.google.com/github/ComplianceAnalytics/aml-book1/blob/main/notebooks/chapter_06.ipynb) | Rule 2 — Transaction Velocity |
| 7 | [chapter_07.ipynb](https://colab.research.google.com/github/ComplianceAnalytics/aml-book1/blob/main/notebooks/chapter_07.ipynb) | Rule 3 — High-Risk Country Counterparty |
| 8 | [chapter_08.ipynb](https://colab.research.google.com/github/ComplianceAnalytics/aml-book1/blob/main/notebooks/chapter_08.ipynb) | Isolation Forest Alert Triage |

---
*© Compliance Analytics Ltd. All dataset content is synthetic and fictional. No real customer data is used.*  
*Repository: [github.com/ComplianceAnalytics/aml-book1](https://github.com/ComplianceAnalytics/aml-book1)*